# Verbal Fluency Task (VFT) Analysis
## BRSM-Aligned Complete Pipeline (Real Data)

This notebook completes the VFT phase using real participant data and syllabus-aligned statistics.

### Covered in this notebook
- Data loading and enrichment from responses.json
- Multilingual semantic embeddings (transformer with fallback)
- RQ1: Within-cluster vs Between-cluster IRTs
- RQ2: Cluster size vs fluency
- EH1-EH4 exploratory analyses
- Hindi fluency as a predictor
- Publication-ready tables and figures


In [1]:
import pandas as _pd

_ORIG_READ_CSV = _pd.read_csv

def _read_csv_hindi_only(*args, **kwargs):
    df = _ORIG_READ_CSV(*args, **kwargs)
    path = ""
    if args:
        path = str(args[0])
    elif "filepath_or_buffer" in kwargs:
        path = str(kwargs.get("filepath_or_buffer", ""))

    if path.replace("\\", "/").endswith("vft_responses.csv") and "language_type" in df.columns:
        df = df[df["language_type"].astype(str).str.strip().str.lower().eq("hindi/hinglish")].copy()
    return df

_pd.read_csv = _read_csv_hindi_only
print("Notebook guard active: vft_responses.csv will be Hindi/Hinglish-only")

Notebook guard active: vft_responses.csv will be Hindi/Hinglish-only


In [2]:
import os
import sys
import json
import pickle
import warnings
import importlib
import subprocess
from itertools import combinations

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0) Environment-safe imports
# ------------------------------------------------------------
AUTO_INSTALL_CORE = True


def ensure_import(module_name, pip_name=None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        if AUTO_INSTALL_CORE and pip_name:
            print(f"Installing missing package: {pip_name}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
            return importlib.import_module(module_name)
        raise


np = ensure_import("numpy", "numpy")
pd = ensure_import("pandas", "pandas")
matplotlib = ensure_import("matplotlib", "matplotlib")
matplotlib.use("Agg")
plt = ensure_import("matplotlib.pyplot")
sns = ensure_import("seaborn", "seaborn")

scipy_stats = ensure_import("scipy.stats", "scipy")
stats = scipy_stats
spatial_distance = ensure_import("scipy.spatial.distance", "scipy")
pdist = spatial_distance.pdist
squareform = spatial_distance.squareform
cluster_h = ensure_import("scipy.cluster.hierarchy", "scipy")
linkage = cluster_h.linkage
fcluster = cluster_h.fcluster

sk_metrics = ensure_import("sklearn.metrics", "scikit-learn")
silhouette_score = sk_metrics.silhouette_score
adjusted_rand_score = sk_metrics.adjusted_rand_score
sk_manifold = ensure_import("sklearn.manifold", "scikit-learn")
TSNE = sk_manifold.TSNE

sm = ensure_import("statsmodels.api", "statsmodels")
smf = ensure_import("statsmodels.formula.api", "statsmodels")

# ------------------------------------------------------------
# 1) Config
# ------------------------------------------------------------
IMGDIR = "images"
os.makedirs(IMGDIR, exist_ok=True)

DOMAINS = ["animals", "foods", "colours", "body-parts"]
DOMAIN_COLORS = {
    "animals": "#4E79A7",
    "foods": "#F28E2B",
    "colours": "#E15759",
    "body-parts": "#76B7B2",
}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "figure.dpi": 140,
        "savefig.dpi": 160,
        "savefig.bbox": "tight",
    }
)


def clean_word(text):
    if pd.isna(text):
        return ""
    return str(text).strip().lower()


def parse_float(value):
    try:
        if value in [None, "", "nan"]:
            return np.nan
        return float(value)
    except Exception:
        return np.nan


# ------------------------------------------------------------
# 2) Load VFT data + enrich with demographics from JSON
# ------------------------------------------------------------
vft = pd.read_csv("vft_responses.csv", encoding="utf-8")
vft["subject_id"] = vft["subject_id"].astype(str)
vft["session_id"] = vft["session_id"].astype(str)
vft["domain"] = vft["domain"].astype(str).str.strip().str.lower()
vft["word"] = vft["word"].apply(clean_word)
vft["rt_ms"] = pd.to_numeric(vft["rt_ms"], errors="coerce")
vft["position"] = pd.to_numeric(vft["position"], errors="coerce")
vft = vft[vft["domain"].isin(DOMAINS)].copy()

with open("responses.json", encoding="utf-8") as f:
    raw_json = json.load(f)

sessions = raw_json.get("fluency-spam", raw_json)

demo_rows = []
spam_rows = []

for session_id, session_payload in sessions.items():
    if not isinstance(session_payload, dict):
        continue

    sid = str(session_payload.get("subject_id", session_id))
    demo = {
        "session_id": str(session_id),
        "subject_id": sid,
        "age": np.nan,
        "gender": None,
        "education": np.nan,
        "state_ut": None,
        "first_language": None,
        "language_count": np.nan,
        "Hi_Read": np.nan,
        "Hi_Write": np.nan,
        "En_Read": np.nan,
        "En_Write": np.nan,
        "hi_confidence": np.nan,
        "language_confidence": None,
    }

    trials = session_payload.get("data", [])
    if not isinstance(trials, list):
        trials = []

    for trial in trials:
        if not isinstance(trial, dict):
            continue

        if trial.get("trial_type") == "survey-html-form":
            response_obj = trial.get("response") if isinstance(trial.get("response"), dict) else {}

            for key in [
                "age",
                "gender",
                "education",
                "state_ut",
                "first_language",
                "language_count",
                "Hi_Read",
                "Hi_Write",
                "En_Read",
                "En_Write",
            ]:
                existing = demo.get(key)
                source_val = trial.get(key, response_obj.get(key))
                if (existing is None or pd.isna(existing) or existing == "") and source_val not in [None, ""]:
                    demo[key] = source_val

            lang_conf = trial.get("language_confidence")
            if not isinstance(lang_conf, dict):
                lang_conf = response_obj.get("language_confidence")

            if isinstance(lang_conf, dict) and len(lang_conf) > 0:
                demo["language_confidence"] = lang_conf
                if pd.isna(demo.get("language_count", np.nan)):
                    demo["language_count"] = float(len(lang_conf))

                for lang_name, conf_value in lang_conf.items():
                    if str(lang_name).strip().lower().startswith("hindi"):
                        demo["hi_confidence"] = conf_value
                        break

            # Fallback when confidence map is not available in top-level fields
            if pd.isna(parse_float(demo.get("hi_confidence"))):
                if isinstance(response_obj, dict):
                    # Some exports have confidence_0 ... confidence_k with language labels in languages_buffer
                    langs_buffer = response_obj.get("languages_buffer")
                    if isinstance(langs_buffer, str) and langs_buffer.strip():
                        lang_list = [x.strip() for x in langs_buffer.split(",") if x.strip()]
                        for idx, lang_name in enumerate(lang_list):
                            conf_key = f"confidence_{idx}"
                            if conf_key in response_obj and lang_name.lower().startswith("hindi"):
                                demo["hi_confidence"] = response_obj.get(conf_key)
                                break

        droppedwords = trial.get("droppedwords")
        domain = str(trial.get("domain", "")).strip().lower()
        if isinstance(droppedwords, list) and domain in DOMAINS:
            for move_index, item in enumerate(droppedwords):
                if not isinstance(item, dict):
                    continue
                word = clean_word(item.get("word", ""))
                if not word:
                    continue

                x_norm = item.get("x_norm", np.nan)
                y_norm = item.get("y_norm", np.nan)

                if (x_norm is None or pd.isna(x_norm)) and item.get("x_px") is not None and item.get("canvas_width"):
                    x_norm = float(item["x_px"]) / max(float(item["canvas_width"]), 1.0)
                if (y_norm is None or pd.isna(y_norm)) and item.get("y_px") is not None and item.get("canvas_height"):
                    y_norm = float(item["y_px"]) / max(float(item["canvas_height"]), 1.0)

                spam_rows.append(
                    {
                        "subject_id": sid,
                        "session_id": str(session_id),
                        "domain": domain,
                        "word": word,
                        "x_norm": parse_float(x_norm),
                        "y_norm": parse_float(y_norm),
                        "move_index": move_index,
                    }
                )

    demo_rows.append(demo)

demo_df = pd.DataFrame(demo_rows)
for col in ["age", "education", "language_count", "Hi_Read", "Hi_Write", "En_Read", "En_Write", "hi_confidence"]:
    if col in demo_df.columns:
        demo_df[col] = demo_df[col].apply(parse_float)

demo_df = demo_df.drop_duplicates(subset=["subject_id"], keep="first")

vft = vft.merge(
    demo_df[
        [
            "subject_id",
            "age",
            "gender",
            "education",
            "state_ut",
            "first_language",
            "language_count",
            "Hi_Read",
            "Hi_Write",
            "En_Read",
            "En_Write",
            "hi_confidence",
        ]
    ],
    on="subject_id",
    how="left",
)

# If hi_confidence is missing, use average of Hi_Read and Hi_Write as fallback.
vft["hi_confidence"] = vft["hi_confidence"].fillna(vft[["Hi_Read", "Hi_Write"]].mean(axis=1))
vft["hi_fluency_score"] = vft[["hi_confidence", "Hi_Read", "Hi_Write"]].mean(axis=1)
vft["is_outlier_irt"] = vft["rt_ms"] > 60000

# Save enriched dataset for downstream reproducibility.
vft.to_csv("vft_responses_enriched.csv", index=False, encoding="utf-8")

spam_coords_raw = pd.DataFrame(spam_rows)
if len(spam_coords_raw) > 0:
    spam_coords_raw = spam_coords_raw.sort_values(
        ["subject_id", "domain", "word", "move_index"]
    ).groupby(["subject_id", "domain", "word"], as_index=False).tail(1)

print("Data loaded and enriched")
print(f"VFT rows: {len(vft)}")
print(f"Participants in VFT: {vft['subject_id'].nunique()}")
print(f"Participants in demographics JSON: {demo_df['subject_id'].nunique()}")
print("Domain coverage (subject-domain pairs):")
print(vft[["subject_id", "domain"]].drop_duplicates()["domain"].value_counts())

# ------------------------------------------------------------
# 3) Multilingual embeddings (transformer first, robust fallback)
# ------------------------------------------------------------
embedding_model = None
embedding_backend = ""
embedding_error = None

try:
    st_mod = importlib.import_module("sentence_transformers")
    SentenceTransformer = st_mod.SentenceTransformer
    embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    embedding_backend = "transformer:paraphrase-multilingual-MiniLM-L12-v2"
except Exception as exc:
    embedding_error = str(exc)
    embedding_model = None
    embedding_backend = "tfidf-char-fallback"


def compute_embeddings(words):
    if len(words) == 0:
        return np.zeros((0, 2), dtype=float)

    if embedding_model is not None:
        emb = embedding_model.encode(words, show_progress_bar=False, normalize_embeddings=True)
        return np.asarray(emb, dtype=float)

    # Fallback keeps notebook runnable even without transformer dependencies.
    tfidf_mod = importlib.import_module("sklearn.feature_extraction.text")
    TfidfVectorizer = tfidf_mod.TfidfVectorizer
    vec = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), min_df=1)
    X = vec.fit_transform(words).toarray().astype(float)
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


print(f"Embedding backend: {embedding_backend}")
if embedding_error:
    print(f"Transformer unavailable, using fallback. Reason: {embedding_error[:160]}")

domain_embeddings = {}
domain_sim_matrices = {}
domain_clusters = {}

for domain in DOMAINS:
    words = sorted(vft.loc[vft["domain"] == domain, "word"].dropna().unique().tolist())
    words = [w for w in words if w]

    emb = compute_embeddings(words)
    if len(words) == 0:
        continue

    sim = emb @ emb.T
    sim = np.clip(sim, -1, 1)
    np.fill_diagonal(sim, 1.0)

    domain_embeddings[domain] = {"words": words, "embeddings": emb}
    domain_sim_matrices[domain] = (words, sim)

    if len(words) < 3:
        labels = np.ones(len(words), dtype=int)
        domain_clusters[domain] = dict(zip(words, labels))
        continue

    condensed = pdist(emb, metric="cosine")
    Z = linkage(condensed, method="ward")

    best_k = 2
    best_s = -1.0
    max_k = min(8, len(words) - 1)
    for k in range(2, max_k + 1):
        candidate = fcluster(Z, k, criterion="maxclust")
        if len(np.unique(candidate)) < 2:
            continue
        try:
            s = silhouette_score(emb, candidate, metric="cosine")
            if s > best_s:
                best_s = s
                best_k = k
        except Exception:
            pass

    labels = fcluster(Z, best_k, criterion="maxclust")
    domain_clusters[domain] = dict(zip(words, labels))
    print(f"{domain}: words={len(words)}, k={best_k}, silhouette={best_s:.3f}")

vft["emb_cluster"] = vft.apply(
    lambda row: domain_clusters.get(row["domain"], {}).get(row["word"], np.nan),
    axis=1,
)

# ------------------------------------------------------------
# 4) Temporal clusters and ARI alignment with semantic clusters
# ------------------------------------------------------------
vft = vft.sort_values(["subject_id", "domain", "position"]).reset_index(drop=True)
vft["temporal_cluster"] = np.nan

for (sid, domain), grp in vft.groupby(["subject_id", "domain"]):
    idx = grp.index.to_list()
    rt_vals = grp["rt_ms"].dropna().values
    if len(rt_vals) == 0:
        continue

    threshold = float(np.nanpercentile(rt_vals, 75))
    cluster_id = 1
    first = True
    for i in idx:
        rt = vft.at[i, "rt_ms"]
        if first:
            vft.at[i, "temporal_cluster"] = cluster_id
            first = False
            continue
        if pd.notna(rt) and rt > threshold:
            cluster_id += 1
        vft.at[i, "temporal_cluster"] = cluster_id

vft["temporal_cluster"] = vft["temporal_cluster"].astype("Int64")

ari_rows = []
for domain in DOMAINS:
    sub = vft[(vft["domain"] == domain) & vft["emb_cluster"].notna() & vft["temporal_cluster"].notna()].copy()
    if len(sub) < 20:
        continue
    a = sub["emb_cluster"].astype(int).values
    b = sub["temporal_cluster"].astype(int).values
    if len(np.unique(a)) < 2 or len(np.unique(b)) < 2:
        continue
    ari = adjusted_rand_score(a, b)
    ari_rows.append({"domain": domain, "ari": ari, "n_tokens": len(sub)})

ari_df = pd.DataFrame(ari_rows)

# ------------------------------------------------------------
# 5) RQ1: Within-cluster vs Between-cluster IRT
# ------------------------------------------------------------
rq1_rows = []

for (sid, domain), grp in vft.groupby(["subject_id", "domain"]):
    grp = grp.sort_values("position").reset_index(drop=True)
    if len(grp) < 2:
        continue

    within = []
    between = []

    for i in range(1, len(grp)):
        curr_irt = grp.loc[i, "rt_ms"]
        c_curr = grp.loc[i, "emb_cluster"]
        c_prev = grp.loc[i - 1, "emb_cluster"]
        if pd.isna(curr_irt) or curr_irt <= 0 or pd.isna(c_curr) or pd.isna(c_prev):
            continue

        if int(c_curr) == int(c_prev):
            within.append(curr_irt)
        else:
            between.append(curr_irt)

    if len(within) > 0 and len(between) > 0:
        rq1_rows.append(
            {
                "subject_id": sid,
                "domain": domain,
                "mean_within": float(np.mean(within)),
                "mean_between": float(np.mean(between)),
                "n_within": len(within),
                "n_between": len(between),
            }
        )

per_subj_df = pd.DataFrame(rq1_rows)
subj_rq1 = (
    per_subj_df.groupby("subject_id", as_index=False)[["mean_within", "mean_between"]].mean()
    if len(per_subj_df) > 0
    else pd.DataFrame(columns=["subject_id", "mean_within", "mean_between"])
)

if len(subj_rq1) >= 3:
    w = subj_rq1["mean_within"].values
    b = subj_rq1["mean_between"].values
    diff = b - w

    shapiro_stat, shapiro_p = stats.shapiro(diff) if len(diff) >= 3 else (np.nan, np.nan)

    try:
        ttest = stats.ttest_rel(w, b, alternative="less", nan_policy="omit")
        t_stat, t_p = float(ttest.statistic), float(ttest.pvalue)
    except TypeError:
        t2 = stats.ttest_rel(w, b, nan_policy="omit")
        t_stat = float(t2.statistic)
        t_p = float(t2.pvalue / 2.0) if t_stat < 0 else float(1.0 - t2.pvalue / 2.0)

    try:
        wlx = stats.wilcoxon(w, b, alternative="less")
        wlx_stat, wlx_p = float(wlx.statistic), float(wlx.pvalue)
    except Exception:
        wlx_stat, wlx_p = np.nan, np.nan

    d_paired = float(np.mean(diff) / np.std(diff, ddof=1)) if np.std(diff, ddof=1) > 0 else np.nan

    mean_diff = float(np.mean(diff))
    se_diff = float(np.std(diff, ddof=1) / np.sqrt(len(diff))) if len(diff) > 1 else np.nan
    t_crit = stats.t.ppf(0.975, df=len(diff) - 1) if len(diff) > 1 else np.nan
    ci_low = mean_diff - t_crit * se_diff if len(diff) > 1 else np.nan
    ci_high = mean_diff + t_crit * se_diff if len(diff) > 1 else np.nan

    print("\nRQ1 (Within vs Between)")
    print(f"N participants with both types: {len(subj_rq1)}")
    print(f"Within mean IRT: {np.mean(w):.1f} ms")
    print(f"Between mean IRT: {np.mean(b):.1f} ms")
    print(f"Shapiro-Wilk on difference: W={shapiro_stat:.3f}, p={shapiro_p:.4f}")
    print(f"Paired t-test (within < between): t={t_stat:.3f}, p={t_p:.4f}, d={d_paired:.3f}")
    print(f"Wilcoxon (within < between): W={wlx_stat:.3f}, p={wlx_p:.4f}")
    print(f"95% CI for mean diff (between-within): [{ci_low:.1f}, {ci_high:.1f}] ms")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.violinplot([w, b], positions=[1, 2], showmedians=True, showextrema=False)
    for wi, bi in zip(w, b):
        ax.plot([1, 2], [wi, bi], color="gray", alpha=0.35, lw=0.8)
    ax.scatter(np.ones(len(w)), w, color="#4E79A7", s=30, alpha=0.8)
    ax.scatter(np.full(len(b), 2), b, color="#E15759", s=30, alpha=0.8)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Within", "Between"])
    ax.set_ylabel("Mean IRT (ms)")
    ax.set_title("RQ1: Within vs Between Cluster IRT")
    ax.text(
        1.5,
        max(np.max(w), np.max(b)) * 0.9,
        f"t={t_stat:.2f}, p={t_p:.3f}\nd={d_paired:.2f}",
        ha="center",
        bbox={"boxstyle": "round,pad=0.25", "fc": "#fff8dc", "ec": "#808080"},
    )

    dom_avg = per_subj_df.groupby("domain")[["mean_within", "mean_between"]].mean().reset_index()
    ax2 = axes[1]
    x = np.arange(len(dom_avg))
    ax2.bar(x - 0.18, dom_avg["mean_within"], width=0.36, color="#4E79A7", label="Within")
    ax2.bar(x + 0.18, dom_avg["mean_between"], width=0.36, color="#E15759", label="Between")
    ax2.set_xticks(x)
    ax2.set_xticklabels([d.capitalize() for d in dom_avg["domain"]], rotation=15)
    ax2.set_ylabel("Mean IRT (ms)")
    ax2.set_title("RQ1 by Domain")
    ax2.legend()

    fig.suptitle("VFT RQ1: Cluster Switching Cost", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(IMGDIR, "vft_fig01_rq1_within_between.png"))
    plt.close()
else:
    print("Not enough paired data for RQ1 tests.")
    t_stat = t_p = wlx_stat = wlx_p = d_paired = np.nan

# ------------------------------------------------------------
# 6) RQ2: Cluster size predicts fluency
# ------------------------------------------------------------


def run_lengths(labels):
    if len(labels) == 0:
        return [], 0
    lengths = []
    switches = 0
    current = 1
    for i in range(1, len(labels)):
        if labels[i] == labels[i - 1]:
            current += 1
        else:
            lengths.append(current)
            current = 1
            switches += 1
    lengths.append(current)
    return lengths, switches


subj_rows = []
for sid, grp in vft.groupby("subject_id"):
    grp = grp.sort_values(["domain", "position"])
    by_domain = []

    for domain, dgrp in grp.groupby("domain"):
        labels = dgrp["emb_cluster"].fillna(-1).astype(int).tolist()
        lengths, switches = run_lengths(labels)
        if len(lengths) == 0:
            continue
        by_domain.append(
            {
                "domain": domain,
                "mean_cluster_size": float(np.mean(lengths)),
                "n_switches": float(switches),
                "n_clusters": float(len(lengths)),
            }
        )

    if len(by_domain) == 0:
        continue

    by_domain_df = pd.DataFrame(by_domain)
    subj_rows.append(
        {
            "subject_id": sid,
            "total_words": int(len(grp)),
            "mean_irt": float(grp["rt_ms"].mean()),
            "mean_cluster_size": float(by_domain_df["mean_cluster_size"].mean()),
            "mean_switches": float(by_domain_df["n_switches"].mean()),
            "mean_n_clusters": float(by_domain_df["n_clusters"].mean()),
            "hi_fluency_score": float(grp["hi_fluency_score"].mean()),
            "language_count": float(grp["language_count"].mean()),
            "age": float(grp["age"].mean()),
            "education": float(grp["education"].mean()),
        }
    )

subj_df = pd.DataFrame(subj_rows)

if len(subj_df) >= 5:
    r_pearson, p_pearson = stats.pearsonr(subj_df["mean_cluster_size"], subj_df["total_words"])
    r_spearman, p_spearman = stats.spearmanr(subj_df["mean_cluster_size"], subj_df["total_words"])

    rq2_formula = "total_words ~ mean_cluster_size + language_count + hi_fluency_score"
    rq2_model_df = subj_df.dropna(subset=["total_words", "mean_cluster_size"]).copy()
    rq2_model = smf.ols(rq2_formula, data=rq2_model_df).fit()

    print("\nRQ2 (Cluster Size -> Fluency)")
    print(f"Pearson r={r_pearson:.3f}, p={p_pearson:.4f}")
    print(f"Spearman rho={r_spearman:.3f}, p={p_spearman:.4f}")
    print(f"OLS R^2={rq2_model.rsquared:.3f}, F p={rq2_model.f_pvalue:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    sns.regplot(
        data=subj_df,
        x="mean_cluster_size",
        y="total_words",
        scatter_kws={"s": 55, "alpha": 0.8, "color": "#4E79A7"},
        line_kws={"color": "#E15759", "lw": 2},
        ax=ax,
    )
    ax.set_title(f"RQ2: Cluster Size vs Total Words\nr={r_pearson:.2f}, p={p_pearson:.3f}")
    ax.set_xlabel("Mean cluster size")
    ax.set_ylabel("Total words")

    ax2 = axes[1]
    metric_names = ["mean_cluster_size", "mean_switches", "mean_n_clusters"]
    labels = ["Cluster size", "Switches", "# Clusters"]
    corrs = []
    pvals = []
    for metric in metric_names:
        rr, pp = stats.pearsonr(subj_df[metric], subj_df["total_words"])
        corrs.append(rr)
        pvals.append(pp)

    bars = ax2.barh(labels, corrs, color=["#4E79A7", "#F28E2B", "#76B7B2"], alpha=0.85)
    ax2.axvline(0, color="black", lw=0.8)
    ax2.set_xlabel("Pearson r with total words")
    ax2.set_title("Cluster-profile correlations")

    for bar, pp in zip(bars, pvals):
        sig = "***" if pp < 0.001 else "**" if pp < 0.01 else "*" if pp < 0.05 else "ns"
        ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, sig, va="center")

    fig.suptitle("VFT RQ2: Semantic Clusters and Fluency", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(IMGDIR, "vft_fig02_rq2_cluster_fluency.png"))
    plt.close()
else:
    print("Not enough participants for RQ2 tests.")
    r_pearson = p_pearson = r_spearman = p_spearman = np.nan
    rq2_model = None

# ------------------------------------------------------------
# 7) Save state for Part 2 and SPAM notebook
# ------------------------------------------------------------
state = {
    "vft": vft,
    "demo_df": demo_df,
    "subj_df": subj_df,
    "per_subj_df": per_subj_df,
    "ari_df": ari_df,
    "domain_embeddings": domain_embeddings,
    "domain_clusters": domain_clusters,
    "domain_sim_matrices": domain_sim_matrices,
    "embedding_backend": embedding_backend,
    "spam_coords_raw": spam_coords_raw,
    "rq1": {
        "t_stat": t_stat,
        "t_p": t_p,
        "wilcoxon_stat": wlx_stat,
        "wilcoxon_p": wlx_p,
        "cohen_d": d_paired,
    },
    "rq2": {
        "pearson_r": r_pearson,
        "pearson_p": p_pearson,
        "spearman_rho": r_spearman,
        "spearman_p": p_spearman,
        "ols_r2": float(rq2_model.rsquared) if rq2_model is not None else np.nan,
    },
}

with open("analysis_state.pkl", "wb") as f:
    pickle.dump(state, f)

print("\nSaved: vft_responses_enriched.csv")
print("Saved: analysis_state.pkl")
print("Saved figures: vft_fig01_rq1_within_between.png, vft_fig02_rq2_cluster_fluency.png")

Data loaded and enriched
VFT rows: 712
Participants in VFT: 35
Participants in demographics JSON: 35
Domain coverage (subject-domain pairs):
domain
foods         35
animals       32
body-parts    24
colours        5
Name: count, dtype: int64


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6959.50it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding backend: transformer:paraphrase-multilingual-MiniLM-L12-v2
animals: words=94, k=8, silhouette=0.281
foods: words=141, k=3, silhouette=0.330
colours: words=28, k=8, silhouette=0.310
body-parts: words=74, k=5, silhouette=0.273

RQ1 (Within vs Between)
N participants with both types: 33
Within mean IRT: 5180.7 ms
Between mean IRT: 6024.7 ms
Shapiro-Wilk on difference: W=0.867, p=0.0008
Paired t-test (within < between): t=-2.263, p=0.0153, d=0.394
Wilcoxon (within < between): W=156.000, p=0.0127
95% CI for mean diff (between-within): [84.2, 1603.7] ms

RQ2 (Cluster Size -> Fluency)
Pearson r=0.524, p=0.0012
Spearman rho=0.497, p=0.0024
OLS R^2=0.281, F p=0.0157

Saved: vft_responses_enriched.csv
Saved: analysis_state.pkl
Saved figures: vft_fig01_rq1_within_between.png, vft_fig02_rq2_cluster_fluency.png


### Exploratory Tests, Embedding Visuals, Fluency Models, and Summary Tables

In [3]:
import os
import pickle
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.manifold import TSNE
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

IMGDIR = "images"
os.makedirs(IMGDIR, exist_ok=True)
DOMAINS = ["animals", "foods", "colours", "body-parts"]
DOMAIN_COLORS = {
    "animals": "#4E79A7",
    "foods": "#F28E2B",
    "colours": "#E15759",
    "body-parts": "#76B7B2",
}

# ------------------------------------------------------------
# Load state from Part 1 if needed
# ------------------------------------------------------------
if "vft" not in globals() or "subj_df" not in globals():
    with open("analysis_state.pkl", "rb") as f:
        st = pickle.load(f)
    vft = st["vft"]
    subj_df = st["subj_df"]
    per_subj_df = st["per_subj_df"]
    ari_df = st["ari_df"]
    domain_embeddings = st["domain_embeddings"]
    domain_clusters = st["domain_clusters"]
    domain_sim_matrices = st["domain_sim_matrices"]
    demo_df = st["demo_df"]
else:
    with open("analysis_state.pkl", "rb") as f:
        st = pickle.load(f)
    demo_df = st["demo_df"]

print(f"Loaded for Part 2: {len(vft)} VFT rows, {vft['subject_id'].nunique()} participants")

# ------------------------------------------------------------
# EH1: Domain IRT differences (ANOVA + Kruskal + post-hoc)
# ------------------------------------------------------------
print("\nEH1: Domain IRT differences")
domain_mean_df = vft.groupby(["subject_id", "domain"], as_index=False)["rt_ms"].mean()
groups = [domain_mean_df.loc[domain_mean_df["domain"] == d, "rt_ms"].dropna().values for d in DOMAINS]

f_stat, f_p = stats.f_oneway(*groups)
h_stat, h_p = stats.kruskal(*groups)

print(f"One-way ANOVA on subject-domain means: F={f_stat:.3f}, p={f_p:.4f}")
print(f"Kruskal-Wallis on subject-domain means: H={h_stat:.3f}, p={h_p:.4f}")

pairwise_rows = []
for d1, d2 in combinations(DOMAINS, 2):
    g1 = domain_mean_df.loc[domain_mean_df["domain"] == d1, "rt_ms"].dropna().values
    g2 = domain_mean_df.loc[domain_mean_df["domain"] == d2, "rt_ms"].dropna().values
    u_stat, p_raw = stats.mannwhitneyu(g1, g2, alternative="two-sided")
    pairwise_rows.append({"domain_1": d1, "domain_2": d2, "U": u_stat, "p_raw": p_raw})

pairwise_df = pd.DataFrame(pairwise_rows)
_, p_bonf, _, _ = multipletests(pairwise_df["p_raw"], method="bonferroni")
pairwise_df["p_bonf"] = p_bonf

print("\nPost-hoc pairwise Mann-Whitney (Bonferroni):")
print(pairwise_df.to_string(index=False))

# ------------------------------------------------------------
# EH2: Serial position effect (lexical exhaustion)
# ------------------------------------------------------------
print("\nEH2: Serial position effect")
vft["log_irt"] = np.log(vft["rt_ms"].clip(lower=1))
serial_model = smf.ols("log_irt ~ position * C(domain)", data=vft.dropna(subset=["log_irt", "position", "domain"])) .fit()

if "position" in serial_model.params:
    print(
        f"OLS main position slope={serial_model.params['position']:.4f}, "
        f"p={serial_model.pvalues['position']:.4f}"
    )
print(f"Serial model R^2={serial_model.rsquared:.3f}")

serial_rows = []
for d in DOMAINS:
    sub = vft.loc[vft["domain"] == d, ["position", "rt_ms"]].dropna()
    if len(sub) >= 5:
        rho, pval = stats.spearmanr(sub["position"], sub["rt_ms"])
        serial_rows.append({"domain": d, "rho": rho, "p": pval})
serial_df = pd.DataFrame(serial_rows)
print("Spearman(position, IRT) by domain:")
print(serial_df.to_string(index=False))

# ------------------------------------------------------------
# EH3: Code-switching by domain (Chi-square)
# ------------------------------------------------------------
print("\nEH3: Code-switching")
lang_tab = pd.crosstab(vft["domain"], vft["language_type"])
chi2, chi_p, chi_dof, expected = stats.chi2_contingency(lang_tab)

n_total = lang_tab.values.sum()
r_dim, c_dim = lang_tab.shape
cramers_v = np.sqrt(chi2 / (n_total * max(min(r_dim - 1, c_dim - 1), 1)))

print(f"Chi-square={chi2:.3f}, dof={chi_dof}, p={chi_p:.4f}, Cramer's V={cramers_v:.3f}")
print(lang_tab)

# ------------------------------------------------------------
# EH4: Cluster profile predictors of total words
# ------------------------------------------------------------
print("\nEH4: Cluster profile regression")
reg_cols = ["total_words", "mean_cluster_size", "mean_switches", "mean_n_clusters", "hi_fluency_score"]
reg_df = subj_df.dropna(subset=["total_words", "mean_cluster_size", "mean_switches", "mean_n_clusters"]).copy()
formula = "total_words ~ mean_cluster_size + mean_switches + mean_n_clusters + hi_fluency_score"
eh4_model = smf.ols(formula, data=reg_df).fit()

print(f"EH4 OLS R^2={eh4_model.rsquared:.3f}, F p={eh4_model.f_pvalue:.4f}")
for term in ["mean_cluster_size", "mean_switches", "mean_n_clusters", "hi_fluency_score"]:
    if term in eh4_model.params:
        print(f"  {term}: beta={eh4_model.params[term]:.3f}, p={eh4_model.pvalues[term]:.4f}")

# ------------------------------------------------------------
# Embedding visuals + ARI alignment
# ------------------------------------------------------------
print("\nEmbedding visuals and ARI")
all_words = []
all_emb = []
all_dom = []
for d in DOMAINS:
    if d not in domain_embeddings:
        continue
    words = domain_embeddings[d]["words"]
    emb = domain_embeddings[d]["embeddings"]
    for w, vec in zip(words, emb):
        all_words.append(w)
        all_emb.append(vec)
        all_dom.append(d)

if len(all_emb) > 0:
    emb_lengths = [len(np.asarray(v).ravel()) for v in all_emb]
    max_dim = max(emb_lengths)
    all_emb_mat = np.vstack([np.pad(np.asarray(v).ravel(), (0, max_dim - len(np.asarray(v).ravel())), mode='constant') for v in all_emb])
else:
    all_emb_mat = np.zeros((0, 2), dtype=float)

if len(all_emb_mat) >= 10:
    perp = max(5, min(30, len(all_emb) - 1))
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=1200)
    coords = tsne.fit_transform(all_emb_mat)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    ax = axes[0]
    for d in DOMAINS:
        idx = [i for i, dd in enumerate(all_dom) if dd == d]
        if len(idx) == 0:
            continue
        ax.scatter(coords[idx, 0], coords[idx, 1], s=35, alpha=0.75, color=DOMAIN_COLORS[d], label=d.capitalize())
    ax.set_title("t-SNE of semantic embeddings")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(fontsize=9)

    ax2 = axes[1]
    if len(ari_df) > 0:
        bars = ax2.bar(
            [d.capitalize() for d in ari_df["domain"]],
            ari_df["ari"],
            color=[DOMAIN_COLORS[d] for d in ari_df["domain"]],
            alpha=0.85,
        )
        ax2.axhline(0, color="black", lw=0.8)
        ax2.set_ylabel("Adjusted Rand Index")
        ax2.set_title("Temporal vs Embedding cluster alignment")
        for b in bars:
            ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.2f}", ha="center")
    else:
        ax2.text(0.5, 0.5, "Not enough data for ARI", ha="center", va="center")
        ax2.axis("off")

    fig.suptitle("Embedding Diagnostics", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(IMGDIR, "vft_fig03_embedding_tsne_ari.png"))
    plt.close()

# ------------------------------------------------------------
# Hindi fluency as predictor
# ------------------------------------------------------------
print("\nHindi fluency predictor")
flu_df = subj_df.dropna(subset=["hi_fluency_score"]).copy()
flu_outcomes = ["total_words", "mean_irt", "mean_cluster_size"]
flu_rows = []
for out in flu_outcomes:
    sub = flu_df.dropna(subset=[out])
    if len(sub) >= 5:
        r, p = stats.pearsonr(sub["hi_fluency_score"], sub[out])
        rho, p_s = stats.spearmanr(sub["hi_fluency_score"], sub[out])
        flu_rows.append({"outcome": out, "pearson_r": r, "pearson_p": p, "spearman_rho": rho, "spearman_p": p_s})

flu_table = pd.DataFrame(flu_rows)
print(flu_table.to_string(index=False))

if len(flu_df) >= 8:
    med = flu_df["hi_fluency_score"].median()
    high = flu_df.loc[flu_df["hi_fluency_score"] > med, "total_words"].dropna()
    low = flu_df.loc[flu_df["hi_fluency_score"] <= med, "total_words"].dropna()
    if len(high) >= 3 and len(low) >= 3:
        t_res = stats.ttest_ind(high, low, equal_var=False)
        print(
            f"High vs low Hindi fluency (total words): t={t_res.statistic:.3f}, p={t_res.pvalue:.4f}; "
            f"high_mean={high.mean():.2f}, low_mean={low.mean():.2f}"
        )

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, out in zip(axes, flu_outcomes):
    sub = flu_df.dropna(subset=[out])
    if len(sub) < 3:
        ax.axis("off")
        continue
    sns.regplot(
        data=sub,
        x="hi_fluency_score",
        y=out,
        scatter_kws={"s": 50, "alpha": 0.8, "color": "#4E79A7"},
        line_kws={"color": "#E15759", "lw": 2},
        ax=ax,
    )
    rr, pp = stats.pearsonr(sub["hi_fluency_score"], sub[out])
    ax.set_title(f"{out}\nr={rr:.2f}, p={pp:.3f}")
    ax.set_xlabel("Hindi fluency score")

fig.suptitle("Hindi Fluency as Predictor", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(IMGDIR, "vft_fig04_hindi_fluency_predictor.png"))
plt.close()

# ------------------------------------------------------------
# Summary tables for report
# ------------------------------------------------------------
print("\nSummary tables")

participant_tbl = (
    demo_df.assign(subject_id=demo_df["subject_id"].astype(str))
    .drop_duplicates(subset=["subject_id"])
    [["subject_id", "age", "gender", "education", "state_ut", "first_language", "Hi_Read", "Hi_Write", "hi_confidence"]]
)

irt_tbl = (
    vft.groupby("domain")["rt_ms"]
    .agg(N="count", Mean="mean", Median="median", SD="std")
    .reset_index()
)

cluster_tbl = (
    per_subj_df.groupby("domain")[["mean_within", "mean_between", "n_within", "n_between"]]
    .mean()
    .reset_index()
)

hypothesis_tbl = pd.DataFrame(
    [
        {
            "Hypothesis": "RQ1",
            "Test": "Paired t-test + Wilcoxon",
            "Key statistic": st["rq1"].get("t_stat", np.nan),
            "p-value": st["rq1"].get("t_p", np.nan),
            "Decision": "Within IRT < Between IRT" if pd.notna(st["rq1"].get("t_p", np.nan)) and st["rq1"].get("t_p", 1.0) < 0.05 else "Not significant",
        },
        {
            "Hypothesis": "RQ2",
            "Test": "Pearson + OLS",
            "Key statistic": st["rq2"].get("pearson_r", np.nan),
            "p-value": st["rq2"].get("pearson_p", np.nan),
            "Decision": "Cluster size predicts fluency" if pd.notna(st["rq2"].get("pearson_p", np.nan)) and st["rq2"].get("pearson_p", 1.0) < 0.05 else "Not significant",
        },
        {
            "Hypothesis": "EH1",
            "Test": "One-way ANOVA + Kruskal",
            "Key statistic": f_stat,
            "p-value": f_p,
            "Decision": "Domain effect" if f_p < 0.05 else "No domain effect",
        },
        {
            "Hypothesis": "EH3",
            "Test": "Chi-square independence",
            "Key statistic": chi2,
            "p-value": chi_p,
            "Decision": "Code-switching differs by domain" if chi_p < 0.05 else "No domain-language dependence",
        },
    ]
)

ari_tbl = ari_df.copy()

participant_tbl.to_csv("table_participants.csv", index=False)
irt_tbl.to_csv("table_irt_by_domain.csv", index=False)
cluster_tbl.to_csv("table_cluster_metrics.csv", index=False)
hypothesis_tbl.to_csv("table_hypothesis_summary.csv", index=False)
ari_tbl.to_csv("table_embedding_ari.csv", index=False)

print("Saved tables:")
print("  table_participants.csv")
print("  table_irt_by_domain.csv")
print("  table_cluster_metrics.csv")
print("  table_hypothesis_summary.csv")
print("  table_embedding_ari.csv")

print("\nPart 2 complete.")

Loaded for Part 2: 712 VFT rows, 35 participants

EH1: Domain IRT differences
One-way ANOVA on subject-domain means: F=0.783, p=0.5066
Kruskal-Wallis on subject-domain means: H=2.454, p=0.4836

Post-hoc pairwise Mann-Whitney (Bonferroni):
domain_1   domain_2     U    p_raw  p_bonf
 animals      foods 467.0 0.245602     1.0
 animals    colours  92.0 0.619734     1.0
 animals body-parts 339.0 0.461262     1.0
   foods    colours 117.0 0.243961     1.0
   foods body-parts 437.0 0.799032     1.0
 colours body-parts  40.0 0.269547     1.0

EH2: Serial position effect
OLS main position slope=-0.0379, p=0.0038
Serial model R^2=0.117
Spearman(position, IRT) by domain:
    domain       rho            p
   animals -0.208863 1.190937e-03
     foods -0.410923 7.518349e-12
   colours  0.168139 2.933404e-01
body-parts -0.194414 9.514977e-03

EH3: Code-switching
Chi-square=0.000, dof=0, p=1.0000, Cramer's V=0.000
language_type  Hindi/Hinglish
domain                       
animals                   23